# 04 · Sequential evaluation, drift and fusion
Loads a checkpoint and evaluates it the way it will be deployed: ordered test runs, the model fed its **own** estimate.
Compares estimators: absolute head only, naive delta integration, leaky/clamped integration, Kalman fusion of delta (process) and
absolute (measurement), Kalman + rest re-anchoring, and fixed-point integrators (truncation vs rounding) to show numeric drift.
Also sweeps the Kalman noise parameters and plots error versus time since the run started.

In [ ]:
# --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
import os, sys, glob, json
for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
else:
    raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# processed data: output of 01_preprocess_split (same session) or attached as a dataset
candidates = [f'{WORK}/db2_processed'] + sorted(glob.glob('/kaggle/input/*/db2_processed')) + sorted(glob.glob('/kaggle/input/*'))
npz_files = []
for c in candidates:
    npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
    if npz_files: break
assert npz_files, 'no processed data found: run 01_preprocess_split first or attach its output'
print(len(npz_files), 'subjects:', [os.path.basename(os.path.dirname(f)) for f in npz_files])

In [ ]:
ckpts = sorted(glob.glob(f'{WORK}/outputs/*.pt') + glob.glob('/kaggle/input/*/outputs/*.pt') + glob.glob('/kaggle/input/*/*.pt'))
CKPT = os.environ.get('EMG_CKPT', ckpts[-1] if ckpts else None); assert CKPT, 'no checkpoint found: run 02 or attach its output'
device = ec.get_device(); model, cfg, extra = ec.load_checkpoint(CKPT, device); model.eval()
out_dir = f'{WORK}/seq_eval'; os.makedirs(out_dir, exist_ok=True)
print('checkpoint', CKPT, '| phase', extra.get('phase'), '| variant', cfg.variant, cfg.heads, 'cond_prev', cfg.cond_prev)
rest_pose, thr = ec.rest_pose_from_train(npz_files), ec.rest_energy_threshold(npz_files)

In [ ]:
# ---- teacher-forced (true previous angle) vs deployment (own estimate fed back) ----
reports = {}
for fb in ([False, True] if cfg.cond_prev else [True]):
    reports[fb] = ec.evaluate_sequential(model, npz_files, cfg, device, rest_pose=rest_pose, rest_threshold=thr, feedback=fb)
    print(f'--- feedback={fb} ---'); display(pd.DataFrame({k: {m: v[m] for m in ('rmse', 'r2', 'mae')} for k, v in reports[fb].items()}).T.sort_values('rmse'))
rep = reports[True]

In [ ]:
# ---- error versus horizon (how fast each estimator drifts) ----
hz = pd.DataFrame({k: v['rmse_vs_horizon'] for k, v in rep.items()}); display(hz)
hz.plot(marker='o', figsize=(10, 5), title='RMSE vs steps since run start (1 step = 25 ms)'); plt.ylabel('RMSE'); plt.grid(alpha=.3)
plt.savefig(f'{out_dir}/rmse_vs_horizon.png', dpi=120); plt.show()

In [ ]:
# ---- Kalman q/r sweep (needs both heads) ----
if hasattr(model, 'fc_abs') and hasattr(model, 'fc_delta'):
    preds = [ec.predict_run(model, r, cfg, device, feedback=True) for f in npz_files for r in ec.ordered_runs(f, 'test') if len(r['angles']) >= 4]
    truth = np.concatenate([p['angles'] for p in preds])
    qs, rs = [1e-5, 1e-4, 1e-3, 1e-2], [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
    grid = pd.DataFrame(index=[f'q={q:g}' for q in qs], columns=[f'r={r:g}' for r in rs], dtype=float)
    for q in qs:
        for r_ in rs:
            est = np.concatenate([ec.kalman_fuse(p['delta'], p['abs'], p['start'], q, r_, energy=p['energy'], rest_threshold=thr, rest_pose=rest_pose, clamp=(-0.1, 1.1)) for p in preds])
            grid.loc[f'q={q:g}', f'r={r_:g}'] = ec.seq_metrics(est, truth)['rmse']
    display(grid); best = grid.stack().idxmin(); print('best (q, r):', best, '->', round(float(grid.stack().min()), 4))
    plt.imshow(grid.values.astype(float), cmap='viridis'); plt.colorbar(label='RMSE'); plt.xticks(range(len(rs)), grid.columns); plt.yticks(range(len(qs)), grid.index)
    plt.title('Kalman fusion RMSE'); plt.savefig(f'{out_dir}/kalman_sweep.png', dpi=120); plt.show()

In [ ]:
# ---- fixed-point integrator drift (why AP_RND and a wide accumulator matter) ----
r = max((r for f in npz_files for r in ec.ordered_runs(f, 'test')), key=lambda x: len(x['angles']))
p = ec.predict_run(model, r, cfg, device, feedback=True)
if p['delta'] is not None:
    j = 3; n = len(r['angles'])
    plt.figure(figsize=(14, 5)); plt.plot(r['angles'][:, j], 'k', label='truth')
    plt.plot(ec.integrate(p['delta'], p['start'])[:, j], label='float integration')
    for fb, rd in [(7, 'floor'), (7, 'round'), (12, 'floor'), (12, 'round')]:
        plt.plot(ec.fixed_point_integrate(p['delta'], p['start'], fb, rd)[:, j], alpha=.7, label=f'{fb} frac bits, {rd}')
    if p['abs'] is not None: plt.plot(ec.kalman_fuse(p['delta'], p['abs'], p['start'], energy=p['energy'], rest_threshold=thr, rest_pose=rest_pose)[:, j], 'g', lw=2, label='kalman + rest anchor')
    plt.title(f'{ec.JOINT_NAMES[j]}: integrator numerics over {n} steps'); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(f'{out_dir}/fixed_point_drift.png', dpi=120); plt.show()
json.dump({'checkpoint': CKPT, 'reports': {str(k): v for k, v in reports.items()}}, open(f'{out_dir}/sequential_report.json', 'w'), indent=2)